In [ ]:
import os

os.environ["ROBOFLOW_API_KEY"] = ""


In [ ]:
import gc
import importlib
import inspect
import json
import sys
import weakref
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
import torchvision.transforms as T
from PIL import Image
from rfdetr import RFDETRBase
from rfdetr.util.misc import NestedTensor
from supervision.metrics import MeanAveragePrecision
from tqdm import tqdm

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# DATASET_ROOT: the versioned COCO folder downloaded from Roboflow
DATASET_ROOT = Path("path/to/basketball-player-detection-2.v13i.coco")
OUTPUT_DIR   = Path("path/to/output/basketball_multilayer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS           = 50
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 16
LEARNING_RATE    = 1e-4

# ── Multi-layer CKA parameters ────────────────────────────────────────────────
# CKA_LAMBDA            : overall weight for the CKA term (0 = disabled)
# CKA_AUGMENTATION_STR  : augmentation intensity for view B
#                         choices → 'weak' | 'moderate' | 'strong'
# CKA_PROGRESSIVE       : ramp CKA weight from 0 → CKA_LAMBDA over first epochs
# CKA_FOCUS_SMALL_OBJ   : assign higher weights to fine-grained backbone scales
# CKA_TEMPERATURE       : softens the CKA alignment target (higher = softer)
CKA_LAMBDA              = 0.5
CKA_AUGMENTATION_STR    = "moderate"
CKA_PROGRESSIVE         = True
CKA_FOCUS_SMALL_OBJ     = True
CKA_TEMPERATURE         = 1.0

# ── Inference ─────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.5

print("=" * 55)
print("MULTI-LAYER CKA — BASKETBALL CONFIGURATION")
print("=" * 55)
print(f"  Dataset root  : {DATASET_ROOT}")
print(f"  Output dir    : {OUTPUT_DIR}")
print(f"  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print()
print("  CKA settings:")
print(f"    lambda              = {CKA_LAMBDA}")
print(f"    augmentation        = {CKA_AUGMENTATION_STR}")
print(f"    progressive         = {CKA_PROGRESSIVE}")
print(f"    focus small objects = {CKA_FOCUS_SMALL_OBJ}")
print(f"    temperature         = {CKA_TEMPERATURE}")
print("=" * 55)

In [ ]:
for mod in ["rfdetr.models.lwdetr", "rfdetr.models"]:
    if mod in sys.modules:
        del sys.modules[mod]

import rfdetr.models.lwdetr

print(f"lwdetr.py : {rfdetr.models.lwdetr.__file__}")

source = inspect.getsource(rfdetr.models.lwdetr.LWDETR.forward)
if "backbone_features" in source:
    print("✓ backbone_features found in LWDETR.forward()")
else:
    print("✗ backbone_features NOT found — please install the modified lwdetr.py")

# ── Forward-pass shape check ───────────────────────────────────────────────────
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_dbg_model = RFDETRBase().model.model.to(device)

_train_dir = DATASET_ROOT / "train"
_test_img_path = next(
    f for f in _train_dir.iterdir()
    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
_img    = Image.open(_test_img_path).convert("RGB").resize((448, 448))
_tensor = T.ToTensor()(_img).unsqueeze(0).to(device)
_mask   = torch.zeros((1, 448, 448), dtype=torch.bool, device=device)

with torch.no_grad():
    _out = _dbg_model(NestedTensor(_tensor, _mask))

if "backbone_features" in _out:
    print(f"\n✓ backbone_features confirmed (device: {device}):")
    for k, v in _out["backbone_features"].items():
        print(f"  Scale {k}: {list(v.shape)}")
else:
    print("\n✗ backbone_features absent from output dict.")

del _dbg_model, _tensor, _mask, _out

In [ ]:
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem   = props.total_memory / 1024 ** 3
        print(f"  GPU {i} : {props.name}  ({mem:.1f} GB)")

In [ ]:
model   = RFDETRBase()
history = []

def _on_epoch_end(data: dict):
    history.append(data)
    map_val = (
        data.get("test_coco_eval_bbox", [0])[0]
        if "test_coco_eval_bbox" in data
        else data.get("test_map", 0)
    )
    msg = (
        f"  Epoch {data.get('epoch', 0):>3}  |  "
        f"Train: {data.get('train_loss', 0):.4f}  |  "
        f"Val: {data.get('test_loss', 0):.4f}  |  "
        f"mAP: {map_val:.4f}"
    )
    layer_parts = []
    for layer_idx in range(3):
        key = f"train_cka_layer_{layer_idx}"
        if key in data:
            layer_parts.append(f"L{layer_idx}={data[key]:.4f}")
    if layer_parts:
        msg += "  |  CKA " + "  ".join(layer_parts)
    print(msg)

model.callbacks["on_fit_epoch_end"].append(_on_epoch_end)
print("Model initialised — starting training ...\n")

model.train(
    dataset_dir             = str(DATASET_ROOT),
    epochs                  = EPOCHS,
    batch_size              = BATCH_SIZE,
    grad_accum_steps        = GRAD_ACCUM_STEPS,
    lr                      = LEARNING_RATE,
    output_dir              = str(OUTPUT_DIR),
    use_ema                 = True,
    tensorboard             = True,
    early_stopping          = True,
    early_stopping_patience = 10,
    amp                     = True,
    device                  = "cuda",
    num_workers             = 0,
    multi_scale             = False,
    resolution              = 448,
    cka_lambda                = CKA_LAMBDA,
    cka_augmentation_strength = CKA_AUGMENTATION_STR,
    cka_progressive           = CKA_PROGRESSIVE,
    cka_focus_small_objects   = CKA_FOCUS_SMALL_OBJ,
    cka_temperature           = CKA_TEMPERATURE,
)

print(f"\n✅ Training complete  |  Checkpoints → {OUTPUT_DIR}")

In [ ]:
def parse_multilayer_cka_logs(log_path: Path) -> dict:
    records = {
        "epochs": [], "cka_total": [],
        "cka_layer_0": [], "cka_layer_1": [], "cka_layer_2": [],
        "progressive_weights": [],
    }
    with log_path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                continue
            records["epochs"].append(data.get("epoch", 0))
            records["cka_total"].append(data.get("loss_cka_total", 0))
            records["cka_layer_0"].append(data.get("loss_cka_layer_0", 0))
            records["cka_layer_1"].append(data.get("loss_cka_layer_1", 0))
            records["cka_layer_2"].append(data.get("loss_cka_layer_2", 0))
            records["progressive_weights"].append(
                data.get("loss_progressive_weight", 1.0)
            )
    return records


log_path = OUTPUT_DIR / "log.txt"
if not log_path.exists():
    print(f"⚠ log.txt not found at {log_path} — run Cell 3 first.")
else:
    data   = parse_multilayer_cka_logs(log_path)
    epochs = data["epochs"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    ax = axes[0, 0]
    ax.plot(epochs, data["cka_layer_0"], label="Scale 0 — C3 (high res)",
            linewidth=2, marker="o")
    ax.plot(epochs, data["cka_layer_1"], label="Scale 1 — C4 (mid res)",
            linewidth=2, marker="s")
    ax.plot(epochs, data["cka_layer_2"], label="Scale 2 — C5 (low res)",
            linewidth=2, marker="^")
    ax.set_title("Layer-wise CKA Losses", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("CKA Loss")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(epochs, data["cka_total"], color="#e74c3c", linewidth=2, marker="d")
    ax.set_title("Total Multi-Layer CKA Loss", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Total CKA Loss")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(epochs, data["progressive_weights"], color="#9b59b6", linewidth=2)
    ax.set_title("Progressive Weight Schedule", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Weight")
    ax.grid(True, alpha=0.3)

    skip = min(5, len(epochs))
    avg_losses = [
        np.mean(data["cka_layer_0"][skip:]) if len(data["cka_layer_0"]) > skip else 0,
        np.mean(data["cka_layer_1"][skip:]) if len(data["cka_layer_1"]) > skip else 0,
        np.mean(data["cka_layer_2"][skip:]) if len(data["cka_layer_2"]) > skip else 0,
    ]
    ax = axes[1, 1]
    ax.bar(
        ["C3\n(high res)", "C4\n(mid res)", "C5\n(low res)"],
        avg_losses,
        color=["#3498db", "#2ecc71", "#f39c12"],
    )
    ax.set_title(f"Avg Layer CKA Loss (epochs {skip}+)", fontweight="bold")
    ax.set_ylabel("Average CKA Loss")
    ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(
        f"Multi-Layer CKA Analysis — Basketball  (λ={CKA_LAMBDA}, {CKA_AUGMENTATION_STR} aug)",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    save_path = OUTPUT_DIR / "multilayer_cka_analysis.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved → {save_path}")

In [ ]:
def cleanup_gpu(obj=None, verbose: bool = True):
    if not torch.cuda.is_available():
        print("No GPU detected — skipping cleanup.")
        return

    torch.cuda.synchronize()
    if verbose:
        alloc = torch.cuda.memory_allocated() / 1024 ** 2
        resv  = torch.cuda.memory_reserved()  / 1024 ** 2
        print(f"Before  allocated: {alloc:.1f} MB  |  reserved: {resv:.1f} MB")

    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("⚠ Object may still have references elsewhere.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()

    if verbose:
        alloc = torch.cuda.memory_allocated() / 1024 ** 2
        resv  = torch.cuda.memory_reserved()  / 1024 ** 2
        print(f"After   allocated: {alloc:.1f} MB  |  reserved: {resv:.1f} MB")


cleanup_gpu(model, verbose=True)

In [ ]:
CHECKPOINT = OUTPUT_DIR / "checkpoint_best_total.pth"

model = RFDETRBase(pretrain_weights=str(CHECKPOINT))
model.optimize_for_inference()
print(f"✓ Model loaded from: {CHECKPOINT}")

ds = sv.DetectionDataset.from_coco(
    images_directory_path = str(DATASET_ROOT / "test"),
    annotations_path      = str(DATASET_ROOT / "test" / "_annotations.coco.json"),
)
print(f"✓ Test set: {len(ds)} images  |  Classes: {ds.classes}")

In [ ]:
targets, predictions = [], []

for path, _, annotations in tqdm(ds, desc="Evaluating"):
    image      = Image.open(path).convert("RGB")
    detections = model.predict(image, threshold=0)
    targets.append(annotations)
    predictions.append(detections)

metric     = MeanAveragePrecision()
map_result = metric.update(predictions, targets).compute()

print("\n" + "=" * 45)
print(f"TEST RESULTS — Basketball Multi-Layer CKA  (λ={CKA_LAMBDA})")
print("=" * 45)
print(f"  mAP@0.5:0.95 : {map_result.map50_95:.4f}")
print(f"  mAP@0.5      : {map_result.map50:.4f}")
print(f"  mAP@0.75     : {map_result.map75:.4f}")

In [ ]:
path, _, annotations = ds[0]
image      = Image.open(path).convert("RGB")
detections = model.predict(image, threshold=CONFIDENCE_THRESHOLD)

print(f"Image : {Path(path).name}")
print(f"GT    : {len(annotations)} objects")
print(f"Pred  : {len(detections)} detections  (threshold={CONFIDENCE_THRESHOLD})")

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness  = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
palette    = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00",
])

bbox_ann  = sv.BoxAnnotator(color=palette, thickness=thickness)
label_ann = sv.LabelAnnotator(color=palette, text_color=sv.Color.BLACK,
                               text_scale=text_scale)

gt_labels   = [ds.classes[c] for c in annotations.class_id]
pred_labels = [f"{ds.classes[c]} {conf:.2f}"
               for c, conf in zip(detections.class_id, detections.confidence)]

gt_img   = label_ann.annotate(bbox_ann.annotate(image.copy(), annotations), annotations, gt_labels)
pred_img = label_ann.annotate(bbox_ann.annotate(image.copy(), detections),  detections,  pred_labels)

sv.plot_images_grid(
    images    = [gt_img, pred_img],
    grid_size = (1, 2),
    titles    = ["Ground Truth", f"RF-DETR Multi-Layer CKA (conf ≥ {CONFIDENCE_THRESHOLD})"],
)
